Below is a **complete Multi-Query Retriever using LangChain + Hugging Face**, without OpenAI.

```text
User Query
    ↓
Hugging Face LLM
    ↓
Generate multiple queries
    ↓
Chroma Vector Store
    ↓
Retrieve documents for each query
    ↓
Remove duplicates
    ↓
Final Context
```

### 1. Install dependencies

In [ ]:
pip install -U langchain langchain-community langchain-huggingface langchain-chroma chromadb sentence-transformers transformers torch

### 2. Complete code

In [2]:
from langchain_huggingface import (
    HuggingFaceEndpoint,
    HuggingFaceEmbeddings
)

from langchain_chroma import Chroma

from langchain_classic.retrievers.multi_query import MultiQueryRetriever

from langchain_core.documents import Document


# --------------------------------------------------
# 1. Hugging Face LLM
# --------------------------------------------------

llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.3",
    task="text-generation",
    max_new_tokens=512,
    temperature=0.2,
)


# --------------------------------------------------
# 2. Hugging Face Embedding Model
# --------------------------------------------------

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# --------------------------------------------------
# 3. Create sample documents
# --------------------------------------------------

documents = [
    Document(
        page_content="""
        Transformers use self-attention to understand relationships
        between tokens in a sequence.
        """
    ),

    Document(
        page_content="""
        Self-attention uses Query, Key and Value vectors to calculate
        how much attention each token should give to other tokens.
        """
    ),

    Document(
        page_content="""
        Multi-head attention allows a Transformer to learn different
        types of relationships between tokens simultaneously.
        """
    ),

    Document(
        page_content="""
        Positional encoding provides information about the position
        of tokens because the Transformer does not process tokens
        sequentially like an RNN.
        """
    ),

    Document(
        page_content="""
        Transformers are trained using large datasets and are capable
        of learning contextual relationships between words.
        """
    ),
]


# --------------------------------------------------
# 4. Create Chroma Vector Store
# --------------------------------------------------

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="transformer_docs"
)


# --------------------------------------------------
# 5. Create normal Retriever
# --------------------------------------------------

base_retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 3
    }
)


# --------------------------------------------------
# 6. Create Multi-Query Retriever
# --------------------------------------------------

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)


# --------------------------------------------------
# 7. Ask a question
# --------------------------------------------------

query = "How does attention help Transformers understand text?"


# --------------------------------------------------
# 8. Retrieve documents using Multi-Query
# --------------------------------------------------

docs = multi_query_retriever.invoke(query)


# --------------------------------------------------
# 9. Display results
# --------------------------------------------------

print("\nRetrieved Documents:\n")

for i, doc in enumerate(docs, start=1):

    print(f"--- Document {i} ---")

    print(doc.page_content)

    print()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7588.77it/s]


ValueError: Model mistralai/Mistral-7B-Instruct-v0.3 is not supported for task text-generation and provider novita. Supported task: conversational.


### 2. Complete code

```python
from langchain_huggingface import (
    HuggingFaceEndpoint,
    HuggingFaceEmbeddings
)

from langchain_chroma import Chroma

from langchain.retrievers.multi_query import MultiQueryRetriever

from langchain_core.documents import Document


# --------------------------------------------------
# 1. Hugging Face LLM
# --------------------------------------------------

llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.3",
    task="text-generation",
    max_new_tokens=512,
    temperature=0.2,
)


# --------------------------------------------------
# 2. Hugging Face Embedding Model
# --------------------------------------------------

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# --------------------------------------------------
# 3. Create sample documents
# --------------------------------------------------

documents = [
    Document(
        page_content="""
        Transformers use self-attention to understand relationships
        between tokens in a sequence.
        """
    ),

    Document(
        page_content="""
        Self-attention uses Query, Key and Value vectors to calculate
        how much attention each token should give to other tokens.
        """
    ),

    Document(
        page_content="""
        Multi-head attention allows a Transformer to learn different
        types of relationships between tokens simultaneously.
        """
    ),

    Document(
        page_content="""
        Positional encoding provides information about the position
        of tokens because the Transformer does not process tokens
        sequentially like an RNN.
        """
    ),

    Document(
        page_content="""
        Transformers are trained using large datasets and are capable
        of learning contextual relationships between words.
        """
    ),
]


# --------------------------------------------------
# 4. Create Chroma Vector Store
# --------------------------------------------------

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="transformer_docs"
)


# --------------------------------------------------
# 5. Create normal Retriever
# --------------------------------------------------

base_retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 3
    }
)


# --------------------------------------------------
# 6. Create Multi-Query Retriever
# --------------------------------------------------

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)


# --------------------------------------------------
# 7. Ask a question
# --------------------------------------------------

query = "How does attention help Transformers understand text?"


# --------------------------------------------------
# 8. Retrieve documents using Multi-Query
# --------------------------------------------------

docs = multi_query_retriever.invoke(query)


# --------------------------------------------------
# 9. Display results
# --------------------------------------------------

print("\nRetrieved Documents:\n")

for i, doc in enumerate(docs, start=1):

    print(f"--- Document {i} ---")

    print(doc.page_content)

    print()
```

---

# 3. What Happens Internally?

When you execute:

```python
docs = multi_query_retriever.invoke(query)
```

LangChain approximately performs:

```text
                    User Query
                        │
                        ▼
      "How does attention help Transformers?"
                        │
                        ▼
                Hugging Face LLM
                        │
                        ▼
             Generate alternative queries
                        │
        ┌───────────────┼────────────────┐
        ▼               ▼                ▼
     Query 1         Query 2          Query 3
        │               │                │
        ▼               ▼                ▼
     Chroma          Chroma           Chroma
        │               │                │
        ▼               ▼                ▼
     Docs             Docs             Docs
        └───────────────┼────────────────┘
                        ▼
                Combine Documents
                        │
                        ▼
                  Remove Duplicates
                        │
                        ▼
                  Final Documents
```

For example, the Hugging Face model could generate queries similar to:

```text
Query 1:
How does self-attention work in Transformers?

Query 2:
How do Transformers use attention to understand relationships
between tokens?

Query 3:
What role does attention play in Transformer architecture?
```

The exact generated queries depend on the model and prompt.

---

# 4. Hugging Face Is Doing Two Different Jobs Here

This is important.

We use **two Hugging Face models/components**.

### Embedding model

```python
HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
```

Responsible for:

```text
Text
 ↓
Embedding Vector
```

Used by Chroma for semantic retrieval.

---

### LLM

```python
HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.3"
)
```

Responsible for:

```text
Original Query
 ↓
Alternative Queries
```

Therefore:

```text
                 Hugging Face
                      │
          ┌───────────┴───────────┐
          ▼                       ▼
    Embedding Model             LLM
          │                       │
          ▼                       ▼
     Vector Search          Query Generation
```

---

# 5. Hugging Face API Token

If you're using `HuggingFaceEndpoint`, you need a Hugging Face access token.

Set it as an environment variable:

### Windows PowerShell

```powershell
$env:HUGGINGFACEHUB_API_TOKEN="your_token"
```

### Linux/macOS

```bash
export HUGGINGFACEHUB_API_TOKEN="your_token"
```

Or use `.env`:

```text
HUGGINGFACEHUB_API_TOKEN=your_token
```

and:

```python
from dotenv import load_dotenv

load_dotenv()
```

Install:

```bash
pip install python-dotenv
```

---

# 6. Better Production Structure

For the RAG project you're building, I would separate the components:

```text
multi_query_rag/
│
├── app.py
├── requirements.txt
├── .env
│
├── data/
│   └── documents/
│
├── embeddings/
│   └── embedding_model.py
│
├── retriever/
│   └── multi_query.py
│
├── llm/
│   └── huggingface.py
│
└── vectorstore/
    └── chroma.py
```

The important architecture is:

```text
Documents
    ↓
Chunking
    ↓
Hugging Face Embeddings
    ↓
Chroma
    ↓
Base Retriever
    ↓
MultiQueryRetriever
    ↑
    │
Hugging Face LLM
    ↑
    │
User Query
```

### One important distinction

**Multi-Query Retriever does not itself generate the final answer.**

It only improves retrieval:

```text
MultiQueryRetriever
       ↓
Relevant Documents
       ↓
Your Generation LLM
       ↓
Final Answer
```

So if you want a **complete Hugging Face RAG application**, the next layer is:

```text
Hugging Face LLM
        +
MultiQueryRetriever
        +
Chroma
        +
Prompt
        ↓
Complete RAG Chatbot
```

That is the architecture I would use for a production-oriented learning project.
